# Find og aflæs stregkoder på pakker

Projektet handler fortsat om pakker. Her er ML-opgaven at finde **stregkodens placering**, hvorefter ZXing aflæser indholdet.

1. Indlæs et rent originalbillede.
2. Find stregkoden med en YOLOv8s-model, der allerede er trænet til stregkoder.
3. Beskær området i original opløsning og afkod det med ZXing.
4. Gem bokse, kodeindhold og billeder til gennemgang.

Dette er **inference med en prætrænet model**, uden finjustering på vores billeder. Modellen kommer fra [Piero2411](https://huggingface.co/Piero2411/YOLOV8s-Barcode-Detection); afkoderen er [ZXing-C++](https://github.com/zxing-cpp/zxing-cpp).

Vælg Python-miljøet `.venv/bin/python` i denne checkout, og kør cellerne oppefra. Gemte resultater er inkluderet.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'config/barcode_model.json').is_file():
    ROOT = ROOT / 'bachelor-detection'
assert (ROOT / 'config/barcode_model.json').is_file(), 'Åbn notebooken i bachelor-detection-mappen.'
sys.path.insert(0, str(ROOT))
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT / 'data/output/.yolo'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'data/output/.matplotlib'))

import cv2
import torch
import pandas as pd
from ultralytics import YOLO
from IPython.display import display, Image, HTML
from src.detect_barcodes import crop_barcode, decode_crop, main
from src.setup_barcode_model import DEFAULT_WEIGHTS, download_weights

torch.set_num_threads(4)
cv2.setNumThreads(4)
# Kontrollér de lokale vægte. Henter kun modellen, hvis den mangler.
download_weights()
print('Python:', sys.executable)
print('Model klar.')

## 1. Indlæs originalbilledet

Skift filnavnet for at prøve en anden pakke. `IMG_0586.jpeg` har en synlig stregkode. `IMG_0594.jpeg` viser en kasse uden synlig stregkode på den viste side.

Kun visningen nedenfor skaleres. Variablen `image` beholder den fulde opløsning, så afkodningen har alle tilgængelige pixels.

In [ ]:
IMAGE_PATH = ROOT / 'data/raw/real_photos/IMG_0586.jpeg'
CONFIDENCE = 0.25
IMAGE_SIZE = 1280

image = cv2.imread(str(IMAGE_PATH))
assert image is not None, f'Billedet kunne ikke læses: {IMAGE_PATH}'
print('Original opløsning:', image.shape[1], 'x', image.shape[0])

def show_image(pixels, width=600):
    scale = min(1, width / pixels.shape[1])
    preview = cv2.resize(pixels, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    ok, encoded = cv2.imencode('.jpg', preview)
    assert ok
    display(Image(data=encoded.tobytes()))

show_image(image)

## 2. ML finder stregkoden

YOLO returnerer en boks og en confidence-score for hvert fund. Confidence-scoren fortæller ikke, om stregkodens indhold kan aflæses korrekt. Vi behandler kun modellens `barcode`-klasse her; QR-koder er ikke en del af dette forsøg.

In [ ]:
model = YOLO(str(DEFAULT_WEIGHTS), task='detect')
barcode_classes = [i for i, name in model.names.items() if name.lower() == 'barcode']
result = model.predict(image, conf=CONFIDENCE, imgsz=IMAGE_SIZE, iou=0.45,
                       classes=barcode_classes, device='cpu', verbose=False)[0]
print('ML-fund:', len(result.boxes))
show_image(result.plot())

## 3. Aflæs de fundne områder

Vi beskærer i **originalbilledet** med lidt margin omkring stregerne. ZXing prøver at aflæse kodens faktiske indhold. Et ML-fund kan godt være korrekt lokaliseret og stadig være ulæseligt på grund af eksempelvis folder eller uskarphed.

Teksten bevares som en streng, så eventuelle indledende nuller ikke forsvinder. En aflæst værdi bliver ikke automatisk et unikt pakke-ID: det kan også være en fælles emballagekode.

Hvis direkte aflæsning mislykkes, prøves gråtone, tærskling, eventuel opskalering og rotationer på ±15°, ±30° og ±45°. Kodens hjørner transformeres tilbage til originalbilledets koordinater.

In [ ]:
rows = []
for index, box in enumerate(result.boxes, start=1):
    xyxy = box.xyxy[0].tolist()
    crop, crop_xyxy = crop_barcode(image, xyxy)
    codes = decode_crop(crop, crop_xyxy, xyxy)
    show_image(crop)
    if not codes:
        rows.append({'fund': index, 'ML-score': float(box.conf[0]),
                     'status': 'Fundet, men ikke aflæst', 'format': '', 'indhold': ''})
    for code in codes:
        rows.append({'fund': index, 'ML-score': float(box.conf[0]), 'status': 'Aflæst',
                     'format': code['format'], 'indhold': code['text']})
pd.DataFrame(rows, columns=['fund', 'ML-score', 'status', 'format', 'indhold'])

## 4. Kør hele billedmappen og gem resultaterne

Denne celle kører det samme forløb på alle billeder i `data/raw/real_photos`. Den gemmer en HTML-oversigt, udsnit og `barcodes.json`. Skift outputmappen for at bevare et andet forsøg.

Antal fund og antal aflæsninger er observationsantal. Uden manuelt kontrollerede stregkodebokse og kodeværdier er tallene ikke precision, recall eller dokumenteret aflæsningsnøjagtighed.

In [ ]:
OUTPUT = ROOT / 'data/output/barcodes'
main(['--input', str(ROOT / 'data/raw/real_photos'), '--output', str(OUTPUT),
      '--confidence', str(CONFIDENCE), '--imgsz', str(IMAGE_SIZE)])

import json
report = json.loads((OUTPUT / 'barcodes.json').read_text())
pd.DataFrame([{'billede': record['image'], 'ML-fund': len(record['detections']),
               'aflæste fund': sum(bool(d['codes']) for d in record['detections'])}
              for record in report['images']])

## Sammen med pakkedetektion

Det eksisterende pakkeforløb er bevaret i `src/detect_packages.py` og `package_detection.ipynb`. Kør nedenstående i en terminal fra denne mappe for også at tegne pakker og knytte stregkodefund til pakkebokse:

```bash
.venv/bin/python -m src.detect_barcodes --with-packages --output data/output/packages_and_barcodes
```

Stregkoder søges i hele billedet, så en overset pakke ikke automatisk fjerner en mulig kode. Ved flere overlappende pakkebokse gemmes tilknytningen som tvetydig. At genkende en tidligere passage kræver desuden en verificeret, unik forsendelseskode og registrering af separate passager; det implementeres ikke i denne notebook.

## 5. Find også hele fragtlabels

En fragtlabel er hele etiketten med eksempelvis adresse og stregkoder. Funktionen udfører ikke OCR. Før egne annotationer og træning bruger den en **YOLO-World-baseline**, hvis kvalitet skal evalueres. Efter træning kan en fælles model med klasserne `barcode` og `label` bruges.

Sæt `RUN_LABEL_PREVIEW = True` for at køre på billedet ovenfor. Lilla bokse viser labels. Stregkodefund og labels gemmes separat, og geometrisk tilknytning bevarer tvetydige tilfælde.


In [ ]:
RUN_LABEL_PREVIEW = False
if RUN_LABEL_PREVIEW:
    label_output = ROOT / 'data/output/notebook_labels'
    main(['--input', str(IMAGE_PATH), '--with-labels', '--output', str(label_output)])
    display(HTML(f'<a href="{label_output.as_uri()}/index.html">Åbn labeloversigten</a>'))
else:
    print('Labelkørsel er slået fra. Sæt RUN_LABEL_PREVIEW = True for at prøve.')


## 6. Forbered og kontrollér egne data

Supervised træning **kræver annotationer**. Brug CVAT eller et tilsvarende værktøj og eksportér egentlige bokskoordinater i YOLO-format. En manglende `.txt` er uannoteret; en gennemgået, tom `.txt` betyder et negativt billede.

Klasser: `0 = barcode`, `1 = label`. Markér alle synlige eksempler på begge klasser, også ulæselige koder. Hold samme pakke og naboframes i ét datasplit. `groups.csv` kan bruges til automatisk kontrol af grupperne. Skabelonen indeholder ingen opdigtede annotationer.


In [ ]:
from src.barcode_dataset import initialize, check_dataset, dataset_summary
DATASET = ROOT / 'data/output/manual_dataset'
DATA_YAML = DATASET / 'data.yaml'
if not DATASET.exists():
    initialize(DATASET, with_labels=True)
print('Datasæt:', DATA_YAML)
try:
    checked = check_dataset(DATA_YAML)
    display(pd.DataFrame(dataset_summary(checked)['splits']).T)
    print('Grupper kontrolleret:', checked['groups_checked'])
except ValueError as error:
    print('Data er endnu ikke klar:', error)


## 7. Supervised finjustering efter inferensbaselinen

Start fra de eksisterende stregkodevægte. `dry-run` kontrollerer data og viser træningsindstillinger uden at indlæse modellen eller ændre vægte. Rigtig træning er slået fra som standard. Testdata bruges ikke til træning eller valg af bedste model.

Skift `RUN_TRAINING` til `True`, når dataene er gennemgået. Gem hvert forsøg i en ny mappe. `best.pt`, træningskurver og et manifest med data-/modelhashes gemmes under forsøget.


In [ ]:
from src.train_barcodes import main as training_main
RUN_TRAINING = False
TRAINING_OUTPUT = ROOT / 'data/output/training/own_v1'
if RUN_TRAINING:
    training_main(['train', '--data', str(DATA_YAML), '--output', str(TRAINING_OUTPUT),
                   '--epochs', '100', '--batch', '4', '--device', 'cpu'])
else:
    print('Træning er slået fra. Kontrollér først data med:')
    print(f'{sys.executable} -m src.train_barcodes train --data {DATA_YAML} --dry-run')


## 8. IoU, confusion matrix, precision og recall

Confidence er en modelscore, ikke automatisk en dokumenteret sandsynlighed. To forskellige grænser anvendes: **confidence** filtrerer fund, mens **IoU** afgør, om en modelboks overlapper referenceboksen tilstrækkeligt. Match er én-til-én og kræver samme klasse.

- **IoU = overlapareal / samlet areal (intersection / union).** Vises pr. korrekt matchet fund samt som gennemsnit samlet og pr. klasse. Uden match er værdien udefineret. Gennemsnittet gælder kun de korrekte match og skal læses sammen med recall.
- **Precision = TP / (TP + FP):** Hvor stor en andel af modellens fund er korrekte?
- **Recall = TP / (TP + FN):** Hvor stor en andel af referenceobjekterne findes?
- **Confusion matrix:** Faktisk klasse på rækkerne og forudsagt klasse i kolonnerne. Background-kolonnen viser oversete objekter; background-rækken viser falske fund.

Rapporten indeholder også F1, AP/mAP og confidencekurver. Accuracy indgår ikke.
Brug valideringsdata til valg af grænser og testdata til den afsluttende sammenligning. Boksdetektion og korrekt aflæsning af indhold er forskellige mål. Se `docs/training_evaluation.md`.


In [ ]:
from src.evaluate_barcodes import main as evaluation_main
RUN_EVALUATION = False
EVAL_CONFIDENCE = 0.25
EVAL_IOU = 0.5
EVALUATION_WEIGHTS = DEFAULT_WEIGHTS  # Skift til TRAINING_OUTPUT / 'detector/weights/best.pt' efter træning.
EVALUATION_OUTPUT = ROOT / 'data/output/evaluation/notebook_baseline'
if RUN_EVALUATION:
    evaluation_main(['--data', str(DATA_YAML), '--weights', str(EVALUATION_WEIGHTS),
                     '--output', str(EVALUATION_OUTPUT), '--split', 'val',
                     '--confidence', str(EVAL_CONFIDENCE), '--iou', str(EVAL_IOU)])
    measured = json.loads((EVALUATION_OUTPUT / 'evaluation.json').read_text())
    summary_rows = []
    for name, metrics in {'Samlet': measured['overall'], **measured['per_class']}.items():
        overlap = measured['iou_summary']['overall'] if name == 'Samlet' else measured['iou_summary']['per_class'][name]
        summary_rows.append({'klasse': name, **metrics, 'gennemsnitlig IoU': overlap['mean']})
    display(pd.DataFrame(summary_rows).set_index('klasse'))
    display(Image(filename=str(EVALUATION_OUTPUT / 'confusion_matrix.png')))
    display(Image(filename=str(EVALUATION_OUTPUT / 'iou_distribution.png')))
    display(Image(filename=str(EVALUATION_OUTPUT / 'confidence_analysis.png')))
else:
    print('Evaluering afventer gennemgåede referenceannotationer. Ingen målt kvalitet opfindes.')


## 9. Se målene på et syntetisk eksempel

Denne gemte funktionstest viser rapportens IoU, confusion matrix, precision og recall, før egne referenceannotationer er tilgængelige. Billedet og boksene er syntetiske. **Tallene dokumenterer ikke modellens kvalitet på rigtige transportbåndsdata.** Cellen starter ingen model eller træning.


In [ ]:
DEMO_REPORT = ROOT / 'data/output/verification/evaluation_demo'
if (DEMO_REPORT / 'evaluation.json').is_file():
    example = json.loads((DEMO_REPORT / 'evaluation.json').read_text())
    assert example.get('purpose') == 'synthetic_functional_test_only', 'Eksemplet skal være markeret som syntetisk.'
    print('Syntetisk funktionstest — ikke resultater på jeres egne data.')
    summary_rows = []
    for name, metrics in {'Samlet': example['overall'], **example['per_class']}.items():
        overlap = example['iou_summary']['overall'] if name == 'Samlet' else example['iou_summary']['per_class'][name]
        summary_rows.append({'klasse': name, **metrics, 'gennemsnitlig IoU': overlap['mean']})
    display(pd.DataFrame(summary_rows).set_index('klasse'))
    display(Image(filename=str(DEMO_REPORT / 'confusion_matrix.png')))
    display(Image(filename=str(DEMO_REPORT / 'iou_distribution.png')))
else:
    print('Intet gemt syntetisk eksempel. Beregn rigtige mål i afsnit 8, når referenceannotationerne er klar.')
